# Day 05

In [1]:
from itertools import islice

# Read and Parse Data

In [2]:
with open('input.txt') as f:
    data = f.read().split('\n')

In [3]:
def parse_data(data):
    # Divide by empty line
    sections = []
    current_section = []
    for line in data:
        if line.strip() == '':
            if current_section:
                sections.append(current_section)
                current_section = []
        else:
            current_section.append(line)
    if current_section:
        sections.append(current_section)
    return sections

data_parsed = parse_data(data)

## Data Preprocessing

In [4]:
section = data_parsed[1]
section[1:]

['3547471595 1239929038 174680800',
 '3052451552 758183681 481745357',
 '0 1427884524 1775655006',
 '2844087171 549819300 208364381',
 '3767989253 4004864866 5194940',
 '3534196909 1414609838 13274686',
 '1775655006 114264781 435554519',
 '4148908402 4010059806 146058894',
 '2729822390 0 114264781',
 '3773184193 4156118700 138848596',
 '2211209525 3203539530 518612865',
 '3912032789 3767989253 236875613']

In [5]:
def process_data(data):
    data_dict = {}
    for idx, section in enumerate(data):
        if idx == 0:
            seeds = list(map(int, section[0][7:].split(' ')))

        else:
            data_dict[section[0][:-5]] = section[1:]

    return seeds, data_dict

# Part 1

In [6]:
len(data_parsed)

8

In [7]:
# The results will be a list of numbers
# The order is: 
# [seed, soil, fertilizer, water, light, temperature, humidity, location]

seeds, data_dict = process_data(data_parsed)
results = [seeds]
while True:
    idx = len(results) - 1
    to_convert = results[-1]
    name, data = next(islice(data_dict.items(), idx, None))
    print(f"Converting: {name}")
    destination_list = []
    for item in to_convert:
        print(f"    Item: {item}")
        added = False
        for info in data:
            destination, origin, length = tuple(map(int, info.split(' ')))
            if item in range(origin, origin+length):
                print(f"        Found in range: {info}")
                print(f"        {origin<=item}, {item<=origin+length-1}")
                destination_list.append(destination + (item - origin))
                added=True
                break
        if not added:
            print(f"        Not found in any range. Adding {item} instead")
            destination_list.append(item)

    print(f"Completed section")
    results.append(destination_list)

    if len(results) == 8:
        break

Converting: seed-to-soil
    Item: 5844012
        Found in range: 2729822390 0 114264781
        True, True
    Item: 110899473
        Found in range: 2729822390 0 114264781
        True, True
    Item: 1132285750
        Found in range: 3052451552 758183681 481745357
        True, True
    Item: 58870036
        Found in range: 2729822390 0 114264781
        True, True
    Item: 986162929
        Found in range: 3052451552 758183681 481745357
        True, True
    Item: 109080640
        Found in range: 2729822390 0 114264781
        True, True
    Item: 3089574276
        Found in range: 0 1427884524 1775655006
        True, True
    Item: 100113624
        Found in range: 2729822390 0 114264781
        True, True
    Item: 2693179996
        Found in range: 0 1427884524 1775655006
        True, True
    Item: 275745330
        Found in range: 1775655006 114264781 435554519
        True, True
    Item: 2090752257
        Found in range: 0 1427884524 1775655006
        True, True
 

In [8]:
min(results[-1])

825516882

# Part 2

In [9]:
# First idea:
# Start from the lowest interval in location and try to build up to a seeds number


In [13]:
def apply_transform_to_interval(intervals, rules):
    transformed_intervals = []
    for interval in intervals:
        new_interval = []
        start, length = interval
        stop = start + length - 1

        for rule in rules:
            destination, source, size = tuple(map(int, rule.split(' ')))

            # Left lim inside range
            if source <= start < source + size:
                # Compute the transformed starting point
                new_start = destination + (start - source)

                # Right lim also inside range
                if stop < source + size:
                    # It means the full range in contained inside the rule range
                    # Add the new start and the lenght
                    new_interval.extend((new_start, length))

                # Right limit is outside the range
                else:
                    # Add the new start and the size of the rule range
                    new_interval.extend((new_start, size))

                    remainder = [new_start+size, length-size]

                